In [1]:
print("karim")

karim


In [2]:
research_prompt = """
You are a research synthesis agent. Given a research question from a meeting and relevant source material, produce a concise research brief.

Rules:
- Synthesize the answer in 3-5 sentences
- Cite your sources
- If the sources don't fully answer the question, say so
- Keep it factual and relevant to the meeting context

Research question: {question}

Source material:
{sources}

Respond with ONLY valid JSON:
{{
  "summary": "3-5 sentence synthesis answering the question",
  "sources": ["source1 description", "source2 description"]
}}
"""

In [32]:
classifier_prompt = """
You are a meeting utterance classifier. Given a transcript chunk from a meeting, classify it into exactly ONE of these categories:

- decision: A conclusion or agreement reached by participants (e.g., "Let's go with option B", "We've decided to postpone the launch")
- task_commitment: Someone commits to doing something or assigns work (e.g., "I'll handle the deployment", "Can you update the docs by Friday?", "Action item: review the PR")
- research_trigger: A question or topic that needs external information (e.g., "What's the current pricing for AWS Lambda?", "Does anyone know the compliance requirements?")
- discussion: General meeting conversation, updates, opinions, or context-sharing
- off_topic: Social chat, jokes, or content unrelated to meeting objectives

Recent context (last few chunks for reference):
{context}

Current chunk to classify:
Speaker: {speaker}
Text: {text}

Respond with ONLY valid JSON:
{{"classification": "<one of: decision, task_commitment, research_trigger, discussion, off_topic>", "confidence": <float 0.0-1.0>}}
"""

In [3]:
from __future__ import annotations
import json
import logging
from pathlib import Path
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing import Any, Dict, List, Optional
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings


c:\Users\KARIM\anaconda3\envs\iti\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\KARIM\anaconda3\envs\iti\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
tavily_api_key = ""

In [5]:
from __future__ import annotations

import logging

from tavily import TavilyClient


logger = logging.getLogger(__name__)


class TavilySearch:
    """Thin wrapper around the Tavily web-search API."""

    def __init__(self) -> None:
        self.client = TavilyClient(api_key=tavily_api_key)

    async def search(self, query: str, max_results: int = 5) -> list[dict]:
        """Search the web and return simplified results."""
        try:
            response = self.client.search(query=query, max_results=max_results)
            results: list[dict] = []
            for item in response.get("results", []):
                results.append({
                    "title": item.get("title", ""),
                    "url": item.get("url", ""),
                    "content": item.get("content", ""),
                })
            return results
        except Exception:
            logger.exception("Tavily search failed for query: %s", query)
            return []


_tavily_search: TavilySearch | None = None


def get_tavily_search() -> TavilySearch:
    global _tavily_search
    if _tavily_search is None:
        _tavily_search = TavilySearch()
    return _tavily_search


In [6]:
class TranscriptChunk(BaseModel):
    meeting_id: str
    speaker: str
    text: str
    timestamp_start: str
    timestamp_end: str
    minute: int
    topic_cluster: Optional[str] = None
    source_type: str = "live_transcript"

In [7]:
chromadb_host: str = "localhost"
chromadb_port: int = 8000

In [8]:
transcript = """
[00:00:00] Alice: Alright, let's get started. Thanks everyone for joining the weekly sync. Today we need to discuss the new RAG implementation for the Teams bot. Bob, do you want to give us an update?
[00:00:15] Bob: Sure. I've been working on integrating the new vector database. The ingestion pipeline is mostly complete. We are now chunking the meeting transcripts and generating embeddings using the new OpenAI model.
[00:00:30] Charlie: That's great. Did we resolve the issue with the chunk size being too large for the context window?
[00:00:38] Bob: Yes, we adjusted the chunk overlap to 200 tokens and the max chunk size to 1000. It seems to be yielding better retrieval results. I've also updated the RAG service to use these new parameters.
[00:00:55] Alice: Awesome. What about the Teams bot integration? Charlie, I know you were looking into the adaptive cards for displaying the research summaries.
[00:01:05] Charlie: Right. The bot is updated to receive the payload from the graph builder. When the researcher node finishes compiling the information, it sends a webhook to the bot, which then formats it into an adaptive card. I still need to tweak the styling, but the data is flowing.
[00:01:25] Bob: One thing to note – the graph builder currently has a bottleneck in the researcher node when we have multiple concurrent requests. We might need to look into adding some asynchronous processing there.
[00:01:40] Alice: Okay, let's add that to the backlog for the next sprint. For now, let's focus on getting the end-to-end flow working for a single user. Charlie, can you have the adaptive cards ready by tomorrow for testing?
[00:01:55] Charlie: Yes, definitely. I'll push the PR by the end of the day.
[00:02:00] Alice: Perfect. Any other blockers before we wrap up?
[00:02:05] Bob: Nope, all good on my end.
[00:02:08] Charlie: Same here.
[00:02:10] Alice: Thanks team. Let's touch base tomorrow morning.
"""

In [9]:
meeting_id = "meeting_12345"

In [10]:
import re
from typing import List

def parse_timestamp(ts: str) -> int:
    """Convert 00:01:05 → seconds"""
    h, m, s = map(int, ts.split(":"))
    return h * 3600 + m * 60 + s


def transcript_to_chunks(
    transcript: str,
    meeting_id: str,
    offset_seconds: float = 0.0,
) -> List[TranscriptChunk]:

    chunks: List[TranscriptChunk] = []

    pattern = r"\[(\d{2}:\d{2}:\d{2})\]\s*(.*?):\s*(.*)"

    for line in transcript.strip().split("\n"):
        match = re.match(pattern, line)
        if not match:
            continue

        timestamp, speaker, text = match.groups()

        start_sec = parse_timestamp(timestamp)
        abs_start = start_sec + offset_seconds

        chunk = TranscriptChunk(
            meeting_id=meeting_id,
            speaker=speaker,
            text=text.strip(),
            timestamp_start=timestamp,
            timestamp_end=timestamp,  
            minute=int(abs_start // 60),
        )

        chunks.append(chunk)

    return chunks

In [11]:
chunks = transcript_to_chunks(transcript, "meeting_id123")

In [12]:
chunks

[TranscriptChunk(meeting_id='meeting_id123', speaker='Alice', text="Alright, let's get started. Thanks everyone for joining the weekly sync. Today we need to discuss the new RAG implementation for the Teams bot. Bob, do you want to give us an update?", timestamp_start='00:00:00', timestamp_end='00:00:00', minute=0, topic_cluster=None, source_type='live_transcript'),
 TranscriptChunk(meeting_id='meeting_id123', speaker='Bob', text="Sure. I've been working on integrating the new vector database. The ingestion pipeline is mostly complete. We are now chunking the meeting transcripts and generating embeddings using the new OpenAI model.", timestamp_start='00:00:15', timestamp_end='00:00:15', minute=0, topic_cluster=None, source_type='live_transcript'),
 TranscriptChunk(meeting_id='meeting_id123', speaker='Charlie', text="That's great. Did we resolve the issue with the chunk size being too large for the context window?", timestamp_start='00:00:30', timestamp_end='00:00:30', minute=0, topic_c

In [ ]:
from __future__ import annotations

import logging

import chromadb
from langchain_huggingface import HuggingFaceEmbeddings


logger = logging.getLogger(__name__)


class RAGService:
    """ChromaDB-backed retrieval-augmented generation service."""

    def __init__(self) -> None:
        self.chroma = chromadb.PersistentClient(
            path="./chroma_db"
        )
        self.collection = self.chroma.get_or_create_collection("meeting_transcripts")
        
        self.embeddings_client = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

    def _embed(self, text: str) -> list[float]:
        """Generate an embedding using sentence-transformers/all-MiniLM-L6-v2."""
        return self.embeddings_client.embed_query(text)

    def embed_and_store(self, chunk: TranscriptChunk) -> None:
        """Embed a transcript chunk and store it in ChromaDB."""
        try:
            embedding = self._embed(chunk.text)
            doc_id = f"{chunk.meeting_id}_{chunk.timestamp_start}"

            metadata = {
                "meeting_id": chunk.meeting_id,
                "speaker": chunk.speaker,
                "timestamp_start": chunk.timestamp_start,
                "timestamp_end": chunk.timestamp_end,
                "minute": chunk.minute,
                "topic_cluster": chunk.topic_cluster or "",
                "source_type": chunk.source_type,
            }

            self.collection.upsert(
                ids=[doc_id],
                embeddings=[embedding],
                documents=[chunk.text],
                metadatas=[metadata],
            )
            logger.debug("Stored chunk %s in ChromaDB", doc_id)
        except Exception:
            logger.exception("Failed to embed and store chunk")
            raise

    def query(
        self,
        question: str,
        meeting_id: str | None = None,
        speaker: str | None = None,
        minute: int | None = None,
        top_k: int = 5,
    ) -> list[dict]:
        """Two-stage retrieval: filter by metadata, then semantic search."""
        where_filter: dict = {}
        if meeting_id is not None:
            where_filter["meeting_id"] = meeting_id
        if speaker is not None:
            where_filter["speaker"] = speaker
        if minute is not None:
            where_filter["minute"] = minute

        embedding = self._embed(question)

        query_kwargs: dict = {
            "query_embeddings": [embedding],
            "n_results": top_k,
        }
        if where_filter:
            if len(where_filter) == 1:
                query_kwargs["where"] = where_filter
            else:
                query_kwargs["where"] = {
                    "$and": [
                        {k: v} for k, v in where_filter.items()
                    ]
                }

        results = self.collection.query(**query_kwargs)
        print("Query results:", results)
        output: list[dict] = []
        if results and results.get("documents"):
            docs = results["documents"][0]
            metadatas = results["metadatas"][0] if results.get("metadatas") else [{}] * len(docs)
            distances = results["distances"][0] if results.get("distances") else [0.0] * len(docs)

            for doc, meta, dist in zip(docs, metadatas, distances):
                output.append({
                    "text": doc,
                    "metadata": meta,
                    "distance": dist,
                })
        print("Formatted output:", output)
        return output

    def upload_document(self, meeting_id: str, text: str, source_name: str) -> None:
        """Embed and store a supplementary document (not a live transcript chunk)."""
        try:
            embedding = self._embed(text)
            doc_id = f"{meeting_id}_doc_{source_name}"

            metadata = {
                "meeting_id": meeting_id,
                "speaker": "",
                "timestamp_start": "",
                "timestamp_end": "",
                "minute": -1,
                "topic_cluster": "",
                "source_type": "uploaded_document",
            }

            self.collection.upsert(
                ids=[doc_id],
                embeddings=[embedding],
                documents=[text],
                metadatas=[metadata],
            )
            logger.info("Stored uploaded document '%s' for meeting %s", source_name, meeting_id)
        except Exception:
            logger.exception("Failed to upload document '%s'", source_name)
            raise


_rag_service: RAGService | None = None


def get_rag_service() -> RAGService:
    global _rag_service
    if _rag_service is None:
        _rag_service = RAGService()
    return _rag_service


In [14]:
a = get_rag_service()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3893.27it/s]


In [15]:
for chunk in chunks:
    a.embed_and_store(chunk)


In [16]:
b = a.query(
    question="What are the new chunk size and overlap parameters Bob set for the embeddings?",
    top_k=3
)

Query results: {'ids': [['meeting_id123_00:00:38', 'meeting_id123_00:00:30', 'meeting_id123_00:00:00']], 'embeddings': None, 'documents': [["Yes, we adjusted the chunk overlap to 200 tokens and the max chunk size to 1000. It seems to be yielding better retrieval results. I've also updated the RAG service to use these new parameters.", "That's great. Did we resolve the issue with the chunk size being too large for the context window?", "Alright, let's get started. Thanks everyone for joining the weekly sync. Today we need to discuss the new RAG implementation for the Teams bot. Bob, do you want to give us an update?"]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'speaker': 'Bob', 'timestamp_start': '00:00:38', 'topic_cluster': '', 'meeting_id': 'meeting_id123', 'minute': 0, 'timestamp_end': '00:00:38', 'source_type': 'live_transcript'}, {'timestamp_end': '00:00:30', 'source_type': 'live_transcript', 'timestamp_start': '00:00:30', 'sp

In [17]:
b

[{'text': "Yes, we adjusted the chunk overlap to 200 tokens and the max chunk size to 1000. It seems to be yielding better retrieval results. I've also updated the RAG service to use these new parameters.",
  'metadata': {'speaker': 'Bob',
   'timestamp_start': '00:00:38',
   'topic_cluster': '',
   'meeting_id': 'meeting_id123',
   'minute': 0,
   'timestamp_end': '00:00:38',
   'source_type': 'live_transcript'},
  'distance': 1.0820015668869019},
 {'text': "That's great. Did we resolve the issue with the chunk size being too large for the context window?",
  'metadata': {'timestamp_end': '00:00:30',
   'source_type': 'live_transcript',
   'timestamp_start': '00:00:30',
   'speaker': 'Charlie',
   'meeting_id': 'meeting_id123',
   'minute': 0,
   'topic_cluster': ''},
  'distance': 1.3029508590698242},
 {'text': "Alright, let's get started. Thanks everyone for joining the weekly sync. Today we need to discuss the new RAG implementation for the Teams bot. Bob, do you want to give us an

In [ ]:
GROQ_API_KEY = ""

In [19]:
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-120b",
    temperature=0.7,
)


In [54]:
llm.invoke(
    "say,hello karim"
)


AIMessage(content='Hello, Karim! 👋', additional_kwargs={'reasoning_content': 'The user says "say,hello karim". They want a response. There\'s no policy violation. Just respond with "Hello Karim". Probably with friendly tone.'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 76, 'total_tokens': 124, 'completion_time': 0.099199313, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.003008353, 'prompt_tokens_details': None, 'queue_time': 0.01722558, 'total_time': 0.102207666}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef5ee-eeb9-75b3-84a0-78de9f718efb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 48, 'total_tokens': 124, 'output_token_details': {'reasoning': 33}})

In [21]:
async def _extract_question(chunk_text: str) -> str:
    """Use the LLM to extract a concise research question from the chunk."""
    response = await llm.ainvoke([
        HumanMessage(
            content=(
                "Extract a single, concise research question from the following "
                "meeting transcript excerpt. Reply with ONLY the question, nothing else.\n\n"
                f"Excerpt: {chunk_text}"
            )
        )
    ])
    return response.content.strip()

In [35]:
async def classify_chunk(
    chunk: TranscriptChunk,
    recent_chunks: list[TranscriptChunk] | None = None,
) -> ClassifiedChunk:
    context = ""
    if recent_chunks:
        context = "\n".join(
            f"[{c.speaker}]: {c.text}" for c in recent_chunks[-5:]
        )

    prompt = classifier_prompt.format(
        context=context or "(no prior context)",
        speaker=chunk.speaker,
        text=chunk.text,
    )

    response = await llm.ainvoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        logger.warning("Failed to parse classifier output: %s", raw)
        data = {"classification": "discussion", "confidence": 0.5}

    classification = data.get("classification", "discussion")
    if classification not in {e.value for e in ChunkClassification}:
        classification = "discussion"

    return ClassifiedChunk(
        chunk=chunk,
        classification=classification,
        confidence=float(data.get("confidence", 0.5)),
    )

In [41]:
classified =[]
for chunk in chunks:
    chunk_for_schema = TranscriptChunk.model_validate(chunk.model_dump())
    recent = [item.chunk for item in classified[-5:]]
    classified_chunk = await classify_chunk(chunk_for_schema, recent_chunks=recent)
    classified.append(classified_chunk)

In [43]:
classified[0].classification = "research_trigger"
classified[3].classification = "research_trigger"

In [44]:
for i in classified:
    print(i.classification)


research_trigger
discussion
discussion
research_trigger
discussion
task_commitment
discussion
task_commitment
task_commitment
discussion
discussion
discussion
decision


In [46]:
from app.graph.state import MeetingState
state = MeetingState()

In [47]:
from app.models.schemas import *
from app.models.enums import *


In [49]:
state["classified"] = classified

In [ ]:
async def researcher_node(state: MeetingState) -> dict:
    """Research agent: retrieves context from RAG and/or the web, then synthesizes."""
    classified = state.get("classified", [])
    research_chunks = [
        c for c in classified
        if c.classification == ChunkClassification.RESEARCH_TRIGGER.value
    ]
    
    if not research_chunks:
        return {"research": []}

    meeting_id = state.get("meeting_id", "")
    briefs: list[ResearchBrief] = []

    for item in research_chunks:
        try:
            question = await _extract_question(item.chunk.text)
            

            # Stage 1: RAG retrieval
            rag_results = get_rag_service().query(
                question=question,
                meeting_id=meeting_id,
                top_k=5,
            )

            from_rag = True
            good_results = [r for r in rag_results if r["distance"] <= 0.7]

            # Stage 2: fall back to Tavily if RAG insufficient
            if len(good_results) < 2:
                from_rag = False
                web_results = await get_tavily_search().search(query=question, max_results=5)
                sources_text = "\n\n".join(
                    f"[{r['title']}]({r['url']})\n{r['content']}"
                    for r in web_results
                )
            else:
                sources_text = "\n\n".join(
                    f"[RAG result] {r['text']}" for r in good_results
                )

            # Stage 3: synthesize with Claude
            prompt = research_prompt.format(
                question=question,
                sources=sources_text or "(no sources found)",
            )
            response = await llm.ainvoke([HumanMessage(content=prompt)])
            raw = response.content.strip()

            try:
                data = json.loads(raw)
            except json.JSONDecodeError:
                logger.warning("Failed to parse researcher output: %s", raw)
                data = {"summary": raw, "sources": []}

            brief = ResearchBrief(
                query=question,
                summary=data.get("summary", raw),
                sources=data.get("sources", []),
                from_rag=from_rag,
            )
            briefs.append(brief)

        except Exception as e:
            logger.exception("Research failed for chunk: %s", item.chunk.text[:80])
            briefs.append(
                ResearchBrief(
                    query=item.chunk.text[:200],
                    summary=f"Research failed: {e}",
                    sources=[],
                    from_rag=False,
                )
            )

    return {"research": briefs}


In [53]:
b = await researcher_node(state)

Query results: {'ids': [[]], 'embeddings': None, 'documents': [[]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[]], 'distances': [[]]}
Formatted output: []
Query results: {'ids': [[]], 'embeddings': None, 'documents': [[]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[]], 'distances': [[]]}
Formatted output: []


In [55]:
print(b)

{'research': [ResearchBrief(query='How can we effectively implement Retrieval‑Augmented Generation (RAG) in the Teams bot to improve its performance?', summary='To implement Retrieval‑Augmented Generation (RAG) in the Teams bot, first build a clean, well‑structured knowledge base and ingest it into a vector database using chunking and embedding techniques (e.g., via LangChain or Azure AI services). At query time, the bot should semantically retrieve the most relevant document fragments, prepend them to the user prompt, and let the LLM generate a response anchored in those sources, which reduces hallucinations and improves relevance. Continuous evaluation—using a golden dataset, user feedback loops, and retrieval quality metrics—helps refine chunking, embedding models, and prompt design for production‑grade performance. Finally, maintain the pipeline with regular updates to the knowledge store and monitoring of latency and cost to keep the bot responsive in real‑time Teams interactions.

In [56]:
b

{'research': [ResearchBrief(query='How can we effectively implement Retrieval‑Augmented Generation (RAG) in the Teams bot to improve its performance?', summary='To implement Retrieval‑Augmented Generation (RAG) in the Teams bot, first build a clean, well‑structured knowledge base and ingest it into a vector database using chunking and embedding techniques (e.g., via LangChain or Azure AI services). At query time, the bot should semantically retrieve the most relevant document fragments, prepend them to the user prompt, and let the LLM generate a response anchored in those sources, which reduces hallucinations and improves relevance. Continuous evaluation—using a golden dataset, user feedback loops, and retrieval quality metrics—helps refine chunking, embedding models, and prompt design for production‑grade performance. Finally, maintain the pipeline with regular updates to the knowledge store and monitoring of latency and cost to keep the bot responsive in real‑time Teams interactions.